# DreamBooth LoRA Fine-tuning Assignment

**Total Points: 100** (Plus 10 Bonus Points)

In this assignment, you will implement key components of the **DreamBooth** technique using **LoRA (Low-Rank Adaptation)** to fine-tune a Stable Diffusion model.

## Theoretical Background

### 1. DreamBooth
DreamBooth is a fine-tuning technique designed to teach a text-to-image model about a specific subject (e.g., a specific person, dog, or object) using only a few images (typically 3-5).

**Key Concepts:**
*   **Rare Token Identifier**: We bind the subject to a unique identifier in the prompt, often a rare token sequence like "sks". For example, "a photo of a sks dog". This avoids colliding with the model's existing knowledge of "dog".
*   **Prior Preservation**: A major challenge in fine-tuning on small datasets is *overfitting* and *language drift*. The model might learn your "sks dog" perfectly but forget what a generic "dog" looks like.
    *   **Solution**: We use a **Prior Preservation Loss**. We generate images of the generic class (e.g., "a photo of a dog") using the frozen pre-trained model. These are key "Class Images".
    *   **Loss Function**: The total loss is a weighted sum of the reconstruction loss on your specific instance images and the reconstruction loss on the generated class images. This forces the model to retain its prior knowledge.

### 2. LoRA (Low-Rank Adaptation)
Fine-tuning the entire UNet (billions of parameters) is computationally expensive and storage-heavy. **LoRA** offers a parameter-efficient alternative.

**How it works:**
Instead of updating the full weight matrix $W$, LoRA freezes $W$ and injects trainable rank decomposition matrices.
*   Hypothesis: The change in weights $\Delta W$ during adaptation has a low "intrinsic rank".
*   Decomposition: $\Delta W = B \cdot A$, where $B \in \mathbb{R}^{d 	imes r}$ and $A \in \mathbb{R}^{r 	imes k}$. 
    *   $r$ is the **rank**, a hyperparameter (e.g., 4 or 8), which is much smaller than the model dimension $d$.
*   Forward Pass: $h = W_0 x + \Delta W x = W_0 x + B A x$.
*   **Result**: We only train $A$ and $B$, reducing trainable parameters by up to 10,000x compared to full fine-tuning.

---

## Objectives & Grading
1.  **Part 1: Data Preparation** (5 Points)
    *   Setup custom dataset.
2.  **Part 2: Dataset Class** (35 Points)
    *   Implement data loading and tokenization.
3.  **Part 3: Training Loop** (35 Points)
    *   Implement noise sampling and loss computation.
4.  **Part 4: Inference & Analysis** (25 Points)
    *   Experiment with inference parameters.
5.  **Bonus: Prior Preservation** (10 Points)
    *   Generate class images.



## 1. Setup and Installation

We install `diffusers`, `transformers`, `accelerate`, and `peft` (Parameter-Efficient Fine-Tuning).



In [ ]:
# Fix PyTorch version conflicts first
# Uninstall conflicting versions and install compatible PyTorch 2.9.0
import subprocess
import sys

print("Fixing PyTorch version conflicts...")
print("Step 1: Uninstalling conflicting PyTorch packages...")

# Uninstall existing torch, torchvision, torchaudio to avoid conflicts
# Use try-except to handle case where packages might not be installed
try:
    subprocess.check_call([
        sys.executable, "-m", "pip", "uninstall", "-y", 
        "torch", "torchvision", "torchaudio"
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print("✓ Uninstalled existing PyTorch packages")
except subprocess.CalledProcessError:
    print("ℹ No existing PyTorch packages to uninstall (or uninstall failed)")

print("\nStep 2: Installing PyTorch 2.9.0 (this may take a few minutes)...")

# Install compatible PyTorch version
# Try CUDA version first, fallback to CPU if it fails
cuda_available = False
try:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--no-cache-dir",
        "torch==2.9.0", "torchvision==0.24.0", "torchaudio==2.9.0",
        "--index-url", "https://download.pytorch.org/whl/cu118"
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print("✓ Installed PyTorch 2.9.0 with CUDA support")
    cuda_available = True
except (subprocess.CalledProcessError, Exception) as e:
    print(f"ℹ CUDA installation not available, trying CPU version...")
    try:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "--no-cache-dir",
            "torch==2.9.0", "torchvision==0.24.0", "torchaudio==2.9.0",
            "--index-url", "https://download.pytorch.org/whl/cpu"
        ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        print("✓ Installed PyTorch 2.9.0 CPU version")
    except Exception as e2:
        print(f"✗ Installation failed: {e2}")
        raise

print("\nStep 3: Installing other required packages...")
# Install other packages
packages_to_install = [
    "diffusers",
    "transformers", 
    "accelerate",
    "peft",
    "bitsandbytes"
]

for pkg in packages_to_install:
    try:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", pkg
        ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        print(f"✓ Installed {pkg}")
    except Exception as e:
        print(f"⚠ Warning: {pkg} installation had issues: {e}")

# Try to install xformers (optional, may fail on some systems)
try:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "xformers"
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print("✓ Installed xformers (optional)")
except Exception:
    print("ℹ xformers installation skipped (optional package)")

print("\n✓ All packages installed successfully!")
print("⚠ IMPORTANT: Please restart the kernel after installation to ensure all packages are properly loaded.")


In [ ]:
# Verification and Fix Cell
# Run this cell after the installation cell to verify and fix any remaining issues
import subprocess
import sys

print("Verifying installation and fixing any issues...")

# Test torchvision import
torchvision_ok = False
try:
    import torchvision
    # Try to access a torchvision function to ensure it's working
    from torchvision import transforms
    torchvision_ok = True
    print("✓ torchvision is working correctly")
except Exception as e:
    print(f"✗ torchvision has issues: {e}")
    print("Attempting to fix torchvision...")
    # Reinstall torchvision
    try:
        # First uninstall
        subprocess.check_call([
            sys.executable, "-m", "pip", "uninstall", "-y", "torchvision"
        ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        
        # Reinstall matching version
        import torch
        torch_version = torch.__version__
        print(f"Reinstalling torchvision to match torch {torch_version}...")
        
        # Try CUDA first
        try:
            subprocess.check_call([
                sys.executable, "-m", "pip", "install", "--no-cache-dir",
                "torchvision==0.24.0", "--index-url", "https://download.pytorch.org/whl/cu118"
            ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        except:
            # Fallback to CPU
            subprocess.check_call([
                sys.executable, "-m", "pip", "install", "--no-cache-dir",
                "torchvision==0.24.0", "--index-url", "https://download.pytorch.org/whl/cpu"
            ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        
        # Test again
        import torchvision
        from torchvision import transforms
        torchvision_ok = True
        print("✓ torchvision fixed and working")
    except Exception as e2:
        print(f"✗ Could not fix torchvision: {e2}")

# Test transformers import
transformers_ok = False
if torchvision_ok:
    try:
        from transformers import AutoTokenizer, PretrainedConfig
        transformers_ok = True
        print("✓ transformers is working correctly")
    except Exception as e:
        print(f"✗ transformers has issues: {e}")
        print("Attempting to fix transformers...")
        try:
            # Force reinstall transformers
            subprocess.check_call([
                sys.executable, "-m", "pip", "install", "--force-reinstall", "--no-cache-dir",
                "transformers"
            ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            
            # Test again
            from transformers import AutoTokenizer, PretrainedConfig
            transformers_ok = True
            print("✓ transformers fixed and working")
        except Exception as e2:
            print(f"✗ Could not fix transformers: {e2}")

if torchvision_ok and transformers_ok:
    print("\n✓ All packages verified and working!")
    print("You can now proceed with the rest of the notebook.")
else:
    print("\n⚠ Some packages still have issues. Please restart the kernel and run the installation cell again.")



In [ ]:
# Standard library imports
import argparse
import copy
import gc
import logging
import math
import os
import shutil
import warnings
from pathlib import Path

# Core imports
import numpy as np
import torch
import torch.nn.functional as F
import torch.utils.checkpoint

# Handle torchvision import with error checking
try:
    from torchvision import transforms
    TORCHVISION_AVAILABLE = True
except Exception as e:
    print(f"Warning: torchvision import failed: {e}")
    print("Please run the verification cell above to fix this issue.")
    TORCHVISION_AVAILABLE = False
    # Create a dummy transforms module to prevent crashes
    class DummyTransforms:
        Compose = None
        Resize = None
        CenterCrop = None
        RandomCrop = None
        ToTensor = None
        Normalize = None
        InterpolationMode = None
    transforms = DummyTransforms()

# Handle transformers import with error checking
try:
    import transformers
    from transformers import AutoTokenizer, PretrainedConfig
    TRANSFORMERS_AVAILABLE = True
except Exception as e:
    print(f"Error: transformers import failed: {e}")
    print("This is likely due to a torchvision issue. Please run the verification cell above.")
    TRANSFORMERS_AVAILABLE = False
    raise ImportError("transformers cannot be imported. Please fix torchvision first.")

# Other imports
from accelerate import Accelerator
from accelerate.logging import get_logger
from accelerate.utils import ProjectConfiguration, set_seed
from huggingface_hub.utils import insecure_hashlib
from packaging import version
from peft import LoraConfig
from peft.utils import get_peft_model_state_dict, set_peft_model_state_dict
from PIL import Image
from PIL.ImageOps import exif_transpose
from torch.utils.data import Dataset
from tqdm.auto import tqdm

# Diffusers imports
import diffusers
from diffusers import (
    AutoencoderKL,
    DDPMScheduler,
    DiffusionPipeline,
    DPMSolverMultistepScheduler,
    StableDiffusionPipeline,
    UNet2DConditionModel,
)
from diffusers.loaders import LoraLoaderMixin
from diffusers.optimization import get_scheduler
from diffusers.training_utils import _set_state_dict_into_text_encoder, cast_training_params
from diffusers.utils import (
    check_min_version,
    convert_state_dict_to_diffusers,
    convert_unet_state_dict_to_peft,
    is_wandb_available,
)
from diffusers.utils.import_utils import is_xformers_available
from diffusers.utils.torch_utils import is_compiled_module

logger = get_logger(__name__)

# Verify critical imports
if not TORCHVISION_AVAILABLE:
    print("⚠ WARNING: torchvision is not available. Some functionality may not work.")
if not TRANSFORMERS_AVAILABLE:
    print("⚠ WARNING: transformers is not available. The notebook will not work properly.")


## 2. Configuration & Data Preparation (Part 1: 5 Points)

**Instructions:**
1.  **Select a Subject**: Choose a subject (e.g., your pet, a toy, yourself).
2.  **Collect Images**: Take or gather **5 original images** of this subject.
    *   *Requirement*: Images should be **consistent** (same subject). Varying backgrounds is good but keep the subject clearly visible.
3.  **Upload**: Run the cell below to create the folder, then manually upload your images to `./instance_data`.
4.  **Configure**: Update `instance_prompt`, `class_prompt`, and `validation_prompt` in the config below.

**Note on Hyperparameters**:
*   `max_train_steps=1000`: Sufficient for LoRA to learn the subject.
*   `train_text_encoder = True`: We train the text encoder (CLIP) to better learn the prompt association (e.g., "sks cat").



In [ ]:
class TrainingConfig:
    pretrained_model_name_or_path = "runwayml/stable-diffusion-v1-5"
    instance_data_dir = "./instance_data"
    class_data_dir = "./class_data"
    output_dir = "./lora-dreambooth-model"
    
    # STUDENT TODO: Update these prompts for your specific subject
    instance_prompt = "a photo of a sks dog"   # Use a unique token like 'sks'
    class_prompt = "a photo of a dog"          # Generic class prompt
    validation_prompt = "a photo of a sks dog in a bucket" 
    
    seed = 42
    resolution = 512
    center_crop = True
    train_text_encoder = True  # Enable to improve prompt fidelity
    train_batch_size = 1
    sample_batch_size = 4
    num_train_epochs = 10      # Increased for better convergence
    max_train_steps = 1000     
    checkpointing_steps = 250
    gradient_accumulation_steps = 1
    learning_rate = 1e-4
    lr_scheduler = "constant"
    lr_warmup_steps = 0
    use_8bit_adam = True
    mixed_precision = "fp16"
    rank = 4 
    with_prior_preservation = True
    prior_loss_weight = 1.0
    num_class_images = 100
    max_grad_norm = 1.0        # Gradient clipping

    # Validation settings
    num_validation_images = 4
    validation_epochs = 50
    validation_images = None
    
    # Misc
    logging_dir = "logs"
    report_to = "tensorboard"
    allow_tf32 = False
    local_rank = -1
    enable_xformers_memory_efficient_attention = True
    pre_compute_text_embeddings = False 
    tokenizer_max_length = None
    text_encoder_use_attention_mask = False
    class_labels_conditioning = None

args = TrainingConfig()

if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"
    args.mixed_precision = "no"
    print("CUDA not available. Switched mixed_precision to 'no'.")


In [ ]:
import os
from pathlib import Path

if not os.path.exists(args.instance_data_dir):
    os.makedirs(args.instance_data_dir)
    print(f"Created directory {args.instance_data_dir}. Please upload your 5 images here.")

if not os.path.exists(args.class_data_dir):
    os.makedirs(args.class_data_dir)

# Helper function to display a grid of images
def image_grid(imgs, rows, cols):
    assert len(imgs) == rows*cols
    w, h = imgs[0].size
    grid = Image.new('RGB', size=(cols*w, rows*h))
    grid_w, grid_h = grid.size
    for i, img in enumerate(imgs):
        grid.paste(img, box=(i%cols*w, i//cols*h))
    return grid


## Bonus: Prior Preservation (10 Points)

**The Problem**: If we only train on "sks dog", the model might forget what other normal dogs look like.
**The Solution**: We enable `with_prior_preservation`. We must provide generated images of generic dogs in `class_data_dir`. The model will calculate loss on these too, ensuring it still knows how to generate generic dogs.

**Bonus Task**: Write or run the code below to generate `num_class_images` (e.g. 100) using the `class_prompt` using the pre-trained model.
*Hint*: Use `StableDiffusionPipeline` to generate images and save them to `args.class_data_dir`.



In [ ]:
# BONUS TODO: Implement Class Image Generation
# 1. Load the pipeline
# 2. Check if class images exist
# 3. If missing, generate `args.num_class_images`
# 4. Save to `args.class_data_dir`

import os
from pathlib import Path

# Check if class images already exist
existing_class_images = list(Path(args.class_data_dir).glob("*.png")) + list(Path(args.class_data_dir).glob("*.jpg"))
num_existing = len(existing_class_images)

if num_existing < args.num_class_images:
    print(f"Found {num_existing} class images. Generating {args.num_class_images - num_existing} more...")
    
    # Load the pre-trained pipeline
    pipe = StableDiffusionPipeline.from_pretrained(
        args.pretrained_model_name_or_path,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        safety_checker=None,
        requires_safety_checker=False,
    )
    pipe = pipe.to(device)
    
    # Generate class images
    num_images_to_generate = args.num_class_images - num_existing
    for i in tqdm(range(num_images_to_generate), desc="Generating class images"):
        # Generate image using the class prompt
        image = pipe(
            args.class_prompt,
            num_inference_steps=50,
            guidance_scale=7.5,
        ).images[0]
        
        # Save the image
        image_path = Path(args.class_data_dir) / f"class_image_{num_existing + i:05d}.png"
        image.save(image_path)
    
    print(f"Generated {num_images_to_generate} class images. Total: {args.num_class_images}")
    del pipe
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
else:
    print(f"Found {num_existing} class images. No need to generate more.")



## Part 2: Dataset Class (35 Points)

We need a PyTorch `Dataset` to handle our images.

**Detailed Algorithm**:
1.  **Instance Data**: We read an image $x$, resize/crop it to $512 \times 512$, and normalize it to $[-1, 1]$. We also tokenize the prompt (e.g., "a photo of a sks dog") into `input_ids`.
2.  **Class Data (Prior Preservation)**: If enabled, we also load a random image from the `class_data_dir` and tokenize the class prompt (e.g., "a photo of a dog").
3.  **Return**: The dataset `__getitem__` returns a dictionary containing both instance and (optional) class examples.

**Task**: Complete the `__getitem__` method.



In [ ]:
def tokenize_prompt(tokenizer, prompt, tokenizer_max_length=None):
    if tokenizer_max_length is not None:
        max_length = tokenizer_max_length
    else:
        max_length = tokenizer.model_max_length

    text_inputs = tokenizer(
        prompt,
        truncation=True,
        padding="max_length",
        max_length=max_length,
        return_tensors="pt",
    )

    return text_inputs


In [ ]:
class DreamBoothDataset(Dataset):
    def __init__(
        self,
        instance_data_root,
        instance_prompt,
        tokenizer,
        class_data_root=None,
        class_prompt=None,
        class_num=None,
        size=512,
        center_crop=False,
        encoder_hidden_states=None,
        class_prompt_encoder_hidden_states=None,
        tokenizer_max_length=None,
    ):
        self.size = size
        self.center_crop = center_crop
        self.tokenizer = tokenizer
        self.encoder_hidden_states = encoder_hidden_states
        self.class_prompt_encoder_hidden_states = class_prompt_encoder_hidden_states
        self.tokenizer_max_length = tokenizer_max_length

        self.instance_data_root = Path(instance_data_root)
        if not self.instance_data_root.exists():
            raise ValueError("Instance images root doesn't exists.")

        self.instance_images_path = list(Path(instance_data_root).iterdir())
        self.num_instance_images = len(self.instance_images_path)
        self.instance_prompt = instance_prompt
        self._length = self.num_instance_images

        if class_data_root is not None:
            self.class_data_root = Path(class_data_root)
            self.class_data_root.mkdir(parents=True, exist_ok=True)
            self.class_images_path = list(self.class_data_root.iterdir())
            if class_num is not None:
                self.num_class_images = min(len(self.class_images_path), class_num)
            else:
                self.num_class_images = len(self.class_images_path)
            self._length = max(self.num_class_images, self.num_instance_images)
            self.class_prompt = class_prompt
        else:
            self.class_data_root = None

        self.image_transforms = transforms.Compose(
            [
                transforms.Resize(size, interpolation=transforms.InterpolationMode.BILINEAR),
                transforms.CenterCrop(size) if center_crop else transforms.RandomCrop(size),
                transforms.ToTensor(),
                transforms.Normalize([0.5], [0.5]),
            ]
        )

    def __len__(self):
        return self._length

    def __getitem__(self, index):
        example = {}
        
        # STUDENT TODO: Implement Instance Loading
        # 1. Open image from self.instance_images_path
        # 2. Apply self.image_transforms
        # 3. Tokenize self.instance_prompt
        
        # Get instance image (cycle through if index exceeds available images)
        instance_image_path = self.instance_images_path[index % self.num_instance_images]
        instance_image = Image.open(instance_image_path).convert("RGB")
        instance_image = exif_transpose(instance_image)  # Handle EXIF orientation
        
        # Apply transforms (resize, crop, normalize to [-1, 1])
        example["instance_images"] = self.image_transforms(instance_image)
        
        # Tokenize instance prompt
        instance_text_inputs = tokenize_prompt(self.tokenizer, self.instance_prompt, self.tokenizer_max_length)
        example["instance_prompt_ids"] = instance_text_inputs.input_ids.squeeze(0)

        if self.class_data_root:
             # STUDENT TODO: Prior Preservation Loading
             # 1. Open image from self.class_images_path
             # 2. Apply same transforms
             # 3. Tokenize self.class_prompt
             
             # Get class image (cycle through if index exceeds available images)
             class_image_path = self.class_images_path[index % self.num_class_images]
             class_image = Image.open(class_image_path).convert("RGB")
             class_image = exif_transpose(class_image)  # Handle EXIF orientation
             
             # Apply same transforms
             example["class_images"] = self.image_transforms(class_image)
             
             # Tokenize class prompt
             class_text_inputs = tokenize_prompt(self.tokenizer, self.class_prompt, self.tokenizer_max_length)
             example["class_prompt_ids"] = class_text_inputs.input_ids.squeeze(0)
            
        return example


In [ ]:
def collate_fn(examples, with_prior_preservation=False):
    has_attention_mask = "instance_attention_mask" in examples[0]

    input_ids = [example["instance_prompt_ids"] for example in examples]
    pixel_values = [example["instance_images"] for example in examples]

    if has_attention_mask:
        attention_mask = [example["instance_attention_mask"] for example in examples]

    if with_prior_preservation:
        input_ids += [example["class_prompt_ids"] for example in examples]
        pixel_values += [example["class_images"] for example in examples]
        if has_attention_mask:
            attention_mask += [example["class_attention_mask"] for example in examples]

    pixel_values = torch.stack(pixel_values)
    pixel_values = pixel_values.to(memory_format=torch.contiguous_format).float()

    input_ids = torch.cat(input_ids, dim=0)

    batch = {
        "input_ids": input_ids,
        "pixel_values": pixel_values,
    }

    if has_attention_mask:
        batch["attention_mask"] = attention_mask

    return batch


## 5. Model Loading

We load the `StableDiffusionPipeline` components.

### Understanding Stable Diffusion
Stable Diffusion is a **Latent Diffusion Model (LDM)**. Unlike standard diffusion models that operate in pixel space (which is computationally expensive), LDMs operate in a compressed **latent space**.

**Architecture Components:**
1.  **VAE (Variational Autoencoder)**:
    *   **Encoder**: Compresses the image $x$ into a smaller latent representation $z = \mathcal{E}(x)$ (e.g. $512 \times 512 \rightarrow 64 \times 64$).
    *   **Decoder**: Reconstructs the image from latents $\hat{x} = \mathcal{D}(z)$.
    *   *Role*: Moves the diffusion process from high-dimensional pixels to low-dimensional latents.
2.  **UNet**:
    *   The core "denoising" engine. It takes a noisy latent $z_t$, the timestep $t$, and text embeddings $c$ as input, and predicts the noise $\epsilon$.
    *   *Role*: Learns to reverse the diffusion process.
3.  **Text Encoder (CLIP)**:
    *   Tokenizes and encodes the text prompt into embeddings.
    *   *Role*: Guides the UNet via cross-attention layers.
4.  **Scheduler**:
    *   Controls the noise schedule (how much noise is added at each step) and the sampling algorithm (how to subtract noise during inference).

**Diagram:**

![Stable Diffusion Architecture](sd_architecture.png)

### LoRA Setup
We create a `LoraConfig`. Note the `target_modules` (query, value, key projections in attention layers). We attach these small adapters to the frozen UNet.



In [ ]:
def import_model_class_from_model_name_or_path(pretrained_model_name_or_path: str, revision: str):
    text_encoder_config = PretrainedConfig.from_pretrained(
        pretrained_model_name_or_path,
        subfolder="text_encoder",
        revision=revision,
    )
    model_class = text_encoder_config.architectures[0]
    if model_class == "CLIPTextModel":
        from transformers import CLIPTextModel
        return CLIPTextModel
    else:
        raise ValueError(f"{model_class} is not supported.")

def encode_prompt(text_encoder, input_ids, attention_mask, text_encoder_use_attention_mask=None):
    text_input_ids = input_ids.to(text_encoder.device)

    if text_encoder_use_attention_mask:
        attention_mask = attention_mask.to(text_encoder.device)
    else:
        attention_mask = None

    prompt_embeds = text_encoder(
        text_input_ids,
        attention_mask=attention_mask,
        return_dict=False,
    )
    prompt_embeds = prompt_embeds[0]
    return prompt_embeds


## Part 3: Training Loop (35 Points)

**Algorithm Detail**:
Standard Diffusion Training:
1.  **Latents**: Encode image $x$ to latent $z_0 = \mathcal{E}(x)$.
2.  **Noise**: Sample gaussian noise $\epsilon \sim \mathcal{N}(0, I)$ and timestep $t$.
3.  **Forward Diffusion**: Add noise to get $z_t = \sqrt{\bar{\alpha}_t}z_0 + \sqrt{1-\bar{\alpha}_t}\epsilon$.
4.  **Prediction**: The model predicts the noise: $\epsilon_\theta(z_t, t, c)$, where $c$ is the text embedding.
5.  **Loss**: $L = ||\epsilon - \epsilon_\theta(z_t, t, c)||^2$.

**Task**: Implement steps 2, 3, 4, 5 inside the loop.



In [ ]:
def training_loop(args, accelerator, unet, text_encoder, vae, noise_scheduler, optimizer, train_dataloader, lr_scheduler):
    
    weight_dtype = torch.float32
    if accelerator.mixed_precision == "fp16":
        weight_dtype = torch.float16
    elif accelerator.mixed_precision == "bf16":
        weight_dtype = torch.bfloat16

    global_step = 0
    
    progress_bar = tqdm(range(args.max_train_steps), disable=not accelerator.is_local_main_process)
    progress_bar.set_description("Steps")

    for epoch in range(args.num_train_epochs):
        unet.train()
        if args.train_text_encoder:
            text_encoder.train()
        
        for step, batch in enumerate(train_dataloader):
            with accelerator.accumulate(unet):
                pixel_values = batch["pixel_values"].to(dtype=weight_dtype)

                if vae is not None:
                    # Convert images to latent space
                    model_input = vae.encode(pixel_values).latent_dist.sample()
                    model_input = model_input * vae.config.scaling_factor
                else:
                    model_input = pixel_values

                bsz = model_input.shape[0]
                timesteps = torch.randint(
                    0, noise_scheduler.config.num_train_timesteps, (bsz,), device=model_input.device
                )
                timesteps = timesteps.long()

                encoder_hidden_states = encode_prompt(
                    text_encoder,
                    batch["input_ids"],
                    batch.get("attention_mask", None),
                    text_encoder_use_attention_mask=args.text_encoder_use_attention_mask,
                )

                # ==========================================
                # STUDENT TODO: Forward Pass & Loss
                # ==========================================
                
                # 1. Sample noise
                noise = torch.randn_like(model_input)
                
                # 2. Add noise to model_input (forward diffusion process)
                # z_t = sqrt(alpha_bar_t) * z_0 + sqrt(1 - alpha_bar_t) * epsilon
                noisy_model_input = noise_scheduler.add_noise(model_input, noise, timesteps)
                
                # 3. Predict noise (unet)
                # The model predicts the noise epsilon given noisy latents, timestep, and text embeddings
                model_pred = unet(
                    noisy_model_input,
                    timesteps,
                    encoder_hidden_states=encoder_hidden_states,
                ).sample
                
                # 4. Compute Loss (MSE)
                # Handle prior preservation: if with_prior_preservation, batch contains both instance and class examples
                if args.with_prior_preservation:
                    # Split batch: first half is instance, second half is class
                    model_pred_instance, model_pred_class = model_pred.chunk(2)
                    noise_instance, noise_class = noise.chunk(2)
                    model_input_instance, model_input_class = model_input.chunk(2)
                    
                    # Instance loss
                    instance_loss = F.mse_loss(model_pred_instance.float(), noise_instance.float(), reduction="mean")
                    
                    # Class loss (prior preservation)
                    class_loss = F.mse_loss(model_pred_class.float(), noise_class.float(), reduction="mean")
                    
                    # Total loss: weighted combination
                    loss = instance_loss + args.prior_loss_weight * class_loss
                else:
                    # Standard MSE loss without prior preservation
                    loss = F.mse_loss(model_pred.float(), noise.float(), reduction="mean")

                # ==========================================

                accelerator.backward(loss)
                if accelerator.sync_gradients:
                    accelerator.clip_grad_norm_(unet.parameters(), args.max_grad_norm)
                optimizer.step()
                lr_scheduler.step()
                optimizer.zero_grad()

            if accelerator.sync_gradients:
                progress_bar.update(1)
                global_step += 1
                
                if global_step % args.checkpointing_steps == 0:
                    save_path = os.path.join(args.output_dir, f"checkpoint-{global_step}")
                    accelerator.save_state(save_path)

            if global_step >= args.max_train_steps:
                break


## 7. Execution

Initialize and run. Notice the check for `prior_preservation`: if you didn't provide class images, ensure you handle it or the code might warn/skip.



In [ ]:
# Initialize Accelerator
accelerator = Accelerator(
    gradient_accumulation_steps=args.gradient_accumulation_steps,
    mixed_precision=args.mixed_precision,
    log_with=args.report_to,
    project_config=ProjectConfiguration(project_dir=args.output_dir, logging_dir=Path(args.output_dir, args.logging_dir))
)

# Load Tokenizer & Models
tokenizer = AutoTokenizer.from_pretrained(args.pretrained_model_name_or_path, subfolder="tokenizer", use_fast=False)
text_encoder_cls = import_model_class_from_model_name_or_path(args.pretrained_model_name_or_path, None)
noise_scheduler = DDPMScheduler.from_pretrained(args.pretrained_model_name_or_path, subfolder="scheduler")
text_encoder = text_encoder_cls.from_pretrained(args.pretrained_model_name_or_path, subfolder="text_encoder")
vae = AutoencoderKL.from_pretrained(args.pretrained_model_name_or_path, subfolder="vae")
unet = UNet2DConditionModel.from_pretrained(args.pretrained_model_name_or_path, subfolder="unet")

# Freeze base models
vae.requires_grad_(False)
text_encoder.requires_grad_(False)
unet.requires_grad_(False)

# Add LoRA Adapters
unet.enable_xformers_memory_efficient_attention()
unet_lora_config = LoraConfig(
    r=args.rank,
    lora_alpha=args.rank,
    init_lora_weights="gaussian",
    target_modules=["to_k", "to_q", "to_v", "to_out.0", "add_k_proj", "add_v_proj"],
)
unet.add_adapter(unet_lora_config)

# Optimizer
if args.use_8bit_adam:
    import bitsandbytes as bnb
    optimizer_class = bnb.optim.AdamW8bit
else:
    optimizer_class = torch.optim.AdamW

# Optimize UNet and optionally Text Encoder
params_to_optimize = list(filter(lambda p: p.requires_grad, unet.parameters()))
if args.train_text_encoder:
    text_encoder.requires_grad_(True)
    params_to_optimize += list(filter(lambda p: p.requires_grad, text_encoder.parameters()))

optimizer = optimizer_class(params_to_optimize, lr=args.learning_rate, weight_decay=1e-2)

# Prepare
unet, text_encoder, optimizer = accelerator.prepare(unet, text_encoder, optimizer)
weight_dtype = torch.float32
if accelerator.mixed_precision == "fp16":
    weight_dtype = torch.float16
elif accelerator.mixed_precision == "bf16":
    weight_dtype = torch.bfloat16
    
vae.to(accelerator.device, dtype=weight_dtype)
if not args.train_text_encoder:
    text_encoder.to(accelerator.device, dtype=weight_dtype)

# Check Prior Preservation (Keep check logic)
if args.with_prior_preservation:
    if not os.path.exists(args.class_data_dir) or len(list(Path(args.class_data_dir).glob("*"))) == 0:
        print("Warning: Class data directory missing or empty. Disabling prior preservation.")
        args.with_prior_preservation = False
        train_dataloader = None # Re-init loader below

# Data Loaders
train_dataset = DreamBoothDataset(
    instance_data_root=args.instance_data_dir,
    instance_prompt=args.instance_prompt,
    class_data_root=args.class_data_dir if args.with_prior_preservation else None,
    class_prompt=args.class_prompt,
    class_num=args.num_class_images,
    tokenizer=tokenizer,
    size=args.resolution,
    center_crop=args.center_crop,
)

train_dataloader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=args.train_batch_size,
    shuffle=True,
    collate_fn=lambda examples: collate_fn(examples, args.with_prior_preservation),
    num_workers=1,
)

lr_scheduler = get_scheduler(
    args.lr_scheduler,
    optimizer=optimizer,
    num_warmup_steps=args.lr_warmup_steps,
    num_training_steps=args.max_train_steps,
)

train_dataloader, lr_scheduler = accelerator.prepare(train_dataloader, lr_scheduler)

print("Starting training...")
training_loop(args, accelerator, unet, text_encoder, vae, noise_scheduler, optimizer, train_dataloader, lr_scheduler)
print("Training finished!")

if accelerator.is_main_process:
    unet = unet.to(torch.float32)
    unet_lora_state_dict = convert_state_dict_to_diffusers(get_peft_model_state_dict(unet))
    LoraLoaderMixin.save_lora_weights(
        save_directory=args.output_dir,
        unet_lora_layers=unet_lora_state_dict,
    )
    print(f"Model saved to {args.output_dir}")


## Part 4: Inference & Analysis (25 Points)

**Instructions**:
1.  Run the inference code.
2.  **Generate 5 distinct samples** by varying parameters.
    *   **Guidance Scale**: Try 5.0 (softer), 7.5 (standard), 10.0 (sharper). High guidance forces the model to follow closely but can ruin quality.
    *   **Inference Steps**: Try 30, 50, 100.
3.  Display these 5 images and specifically mention the prompt and parameters used for each.



In [ ]:
from diffusers import StableDiffusionPipeline
import torch

model_path = "runwayml/stable-diffusion-v1-5"
lora_path = args.output_dir

# Load Pipeline
pipe = StableDiffusionPipeline.from_pretrained(model_path, torch_dtype=torch.float16)
pipe.load_lora_weights(lora_path)
pipe.to("cuda")

# STUDENT EXPERIMENT: Change these values to generate 5 different samples
prompt = "a photo of a sks dog in a bucket"
image = pipe(prompt, num_inference_steps=50, guidance_scale=5.0).images[0]
image.save("result_1.png")
image
